# Trabalho 5: Privacidade Diferencial – Mecanismo de Resposta Randômica Generalizada (GRR)

Notebook principal: carrega a base, roda as estratégias E1 e E2 para cada ε (10 repetições com sementes fixas) e gera as tabelas e os gráficos, que são salvos em `resultados/`.

Etapas:
1. carrega e codifica o `dados_censo.csv`;
2. calcula as respostas verdadeiras das 10 consultas;
3. define as estratégias (E1 e E2 nas larguras de bin testadas);
4. roda o experimento e calcula o MAE, com e sem truncamento;
5. gera os gráficos e as tabelas;
6. mostra o passo a passo de um usuário (para a apresentação).

O código de cada etapa está nos módulos do projeto (ver `Readme.md`).

In [ ]:
import os
from functools import partial

import numpy as np
import pandas as pd

from contrato import NOMES_CONSULTAS, EPSILONS, REPETICOES, VERDADE_REFERENCIA
from dados import carregar_dados
from consultas import respostas_verdadeiras
from estrategias import estrategia_e1, estrategia_e2
from avaliacao import avaliar, resumo_mae, mae_por_consulta
from graficos import grafico_mae_por_eps, tabela_mae_por_consulta, grafico_larguras_e2

PASTA_RESULTADOS = "resultados"
os.makedirs(PASTA_RESULTADOS, exist_ok=True)

## 1. Dados

In [ ]:
idades, salarios = carregar_dados()
print(f"{len(idades)} registros | idade de {idades.min()} a {idades.max()} | "
      f"{salarios.mean() * 100:.2f}% com >50K")

## 2. Consultas e respostas verdadeiras

In [ ]:
verdade = respostas_verdadeiras(idades, salarios)
assert dict(zip(NOMES_CONSULTAS, verdade)) == VERDADE_REFERENCIA
pd.DataFrame({"consulta": NOMES_CONSULTAS, "verdade": verdade})

## 3. Estratégias

O E1 usa o domínio conjunto de 148 valores. Para o E2, a Pessoa A define as larguras de bin testadas e a largura principal (com a justificativa no `Readme.md`).

In [ ]:
LARGURAS_E2 = []          # Pessoa A: larguras de bin testadas no E2
LARGURA_PRINCIPAL = None  # Pessoa A: largura escolhida

estrategias = {"E1": estrategia_e1}
for w in LARGURAS_E2:
    estrategias[f"E2-w{w}"] = partial(estrategia_e2, largura=w)
list(estrategias)

## 4. Experimento e MAE

Cada estratégia roda 10 vezes para cada ε, com as sementes fixas definidas em `avaliacao.py`.

In [ ]:
df = avaliar(estrategias, idades, salarios, EPSILONS, REPETICOES)
df.to_csv(os.path.join(PASTA_RESULTADOS, "resultados_brutos.csv"), index=False)

resumo = resumo_mae(df, truncado=False)
resumo_trunc = resumo_mae(df, truncado=True)
resumo.to_csv(os.path.join(PASTA_RESULTADOS, "mae_por_epsilon.csv"), index=False)
resumo_trunc.to_csv(os.path.join(PASTA_RESULTADOS, "mae_por_epsilon_truncado.csv"), index=False)
resumo

## 5. Gráficos e tabelas

In [ ]:
EPS_FIXO = 1.0  # ε usado no MAE por consulta (escolha da Pessoa A)

por_consulta = mae_por_consulta(df, EPS_FIXO, truncado=False)
por_consulta.to_csv(os.path.join(PASTA_RESULTADOS, f"mae_por_consulta_eps_{EPS_FIXO}.csv"), index=False)

grafico_mae_por_eps(resumo, os.path.join(PASTA_RESULTADOS, "grafico_mae_por_epsilon.png"))
tabela_mae_por_consulta(por_consulta, os.path.join(PASTA_RESULTADOS, "grafico_mae_por_consulta.png"))
grafico_larguras_e2(resumo, os.path.join(PASTA_RESULTADOS, "grafico_larguras_e2.png"))

## 6. Passo a passo de um usuário (para a apresentação)

Como o `--detalhar` do trabalho 3: escolhe um usuário e mostra o par verdadeiro, o report enviado em cada ε e como o servidor estima as contagens.

In [ ]:
# Pessoa B: demonstração com um usuário (por exemplo, o de índice 0)